# Modeling Business Success in Boston
**DS2500 — Addison Apisarnthanarax**

This notebook predicts storefront business success in Boston using the City of Boston DBA registry (3,721 businesses with GPS coordinates). We use five modeling approaches:

1. **Full Classification** — predicts expired vs active using all features
2. **Entrepreneur Model** — only uses features known before opening (no data leakage)
3. **Longevity Regression** — predicts business age from neighborhood + type
4. **K-Means Clustering** — groups businesses by GPS location
5. **Spatial Model** — adds GPS-derived features to improve prediction

In [ ]:
import sys
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import ConfusionMatrixDisplay

from modeling import (
    load_and_prepare_data, categorize_business, prepare_features,
    train_and_evaluate, prepare_entrepreneur_features,
    ZIPS_BY_NEIGHBORHOOD, BUSINESS_CATEGORIES
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Data Loading & Feature Engineering

In [ ]:
df = load_and_prepare_data('CityofBoston-CityClerkDBA_cleaned.csv')
print(f'Dataset: {len(df)} businesses')
print(f'Active: {(df["is_expired"] == 0).sum()} ({(df["is_expired"] == 0).mean():.1%})')
print(f'Expired: {(df["is_expired"] == 1).sum()} ({(df["is_expired"] == 1).mean():.1%})')
df.head()

In [ ]:
print('Neighborhoods:')
print(df['neighborhood'].value_counts())
print(f'\nBusiness Categories ({df["business_category"].nunique()}):')
print(df['business_category'].value_counts())

## Part 1: Full Classification Model

Uses all available features: **Neighborhood, Business Category, Age, Was Renewed**.

Overfitting prevention:
- `max_depth=5` on tree-based models
- Stratified 80/20 train/test split
- 5-fold cross-validation

In [ ]:
X, y, feature_names, le_neigh, le_cat = prepare_features(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Features: {feature_names}')

results = train_and_evaluate(X_train, X_test, y_train, y_test, feature_names)

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, (name, res) in enumerate(results.items()):
    ConfusionMatrixDisplay(
        confusion_matrix=res['confusion_matrix'],
        display_labels=['Active', 'Expired']
    ).plot(ax=axes.flatten()[idx], cmap='Blues', colorbar=False)
    axes.flatten()[idx].set_title(name)
plt.suptitle('Part 1: Full Model — Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (Random Forest)
rf = results['Random Forest']['model']
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color='steelblue')
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest — Feature Importance')
plt.tight_layout()
plt.show()

print('\nAge and renewal status dominate — but these are known AFTER opening.')
print('This motivates the entrepreneur model (Part 2) which excludes them.')

In [ ]:
# Cross-validation
from sklearn.tree import DecisionTreeClassifier
best_model = DecisionTreeClassifier(max_depth=5, random_state=42)
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='f1')
print(f'5-Fold CV F1: {cv_scores.round(4)}')
print(f'Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'\nTest F1 was {results["Decision Tree"]["f1"]:.4f} — close to CV mean, no overfitting.')

## Part 2: Entrepreneur Model

Only uses **Neighborhood + Business Category** — the features an entrepreneur knows before opening. No age, no renewal (that would be data leakage).

In [ ]:
X_ent, y_ent, feat_ent, _, _ = prepare_entrepreneur_features(df)
X_tr, X_te, y_tr, y_te = train_test_split(X_ent, y_ent, test_size=0.2, random_state=42, stratify=y_ent)
ent_results = train_and_evaluate(X_tr, X_te, y_tr, y_te, feat_ent)

print('\nNeighborhood + type alone barely beat the 70% baseline.')
print('This tells us: location and type are NOT strong standalone predictors.')

In [ ]:
# Survival heatmap — the most actionable visualization
cat_counts = df['business_category'].value_counts()
top_cats = cat_counts[cat_counts >= 30].index
subset = df[df['business_category'].isin(top_cats)]

survival = subset.groupby(['neighborhood', 'business_category'])['is_expired'].apply(
    lambda x: (1 - x.mean()) * 100
).unstack(fill_value=np.nan)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(survival, annot=True, fmt='.0f', cmap='RdYlGn', center=70,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Survival Rate (%)'})
ax.set_title('Business Survival Rate (%) by Neighborhood and Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('While overall prediction is weak, specific combinations show clear patterns.')
print('e.g., Medical in Roxbury: 83% survival vs Hospitality in South Boston: 44%')

## Part 3: Longevity Regression

Predicts **how long a business will last** (age in years) from neighborhood + type.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

le_n, le_c = LabelEncoder(), LabelEncoder()
df_reg = df.copy()
df_reg['n_enc'] = le_n.fit_transform(df_reg['neighborhood'])
df_reg['c_enc'] = le_c.fit_transform(df_reg['business_category'])

X_r = df_reg[['n_enc', 'c_enc']].values
y_r = df_reg['age_years'].values
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

reg_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    'KNN (k=5)': KNeighborsRegressor(n_neighbors=5),
}

for name, model in reg_models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    print(f'{name:20s}  R²={r2_score(y_te, pred):.4f}  MAE={mean_absolute_error(y_te, pred):.2f} years')

print('\nR² near 0 = neighborhood + type explain almost none of the variance.')
print('Business longevity depends on factors not in this dataset (capital, management, etc).')

## Part 4: K-Means Clustering (GPS)

Groups businesses by geographic location to test the hypothesis: **do businesses in denser clusters survive longer?**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

coords = df[['lat', 'lon']].values

# Find optimal k
k_range = range(3, 11)
silhouettes = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(coords)
    sil = silhouette_score(coords, labels)
    silhouettes.append(sil)

best_k = list(k_range)[np.argmax(silhouettes)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), silhouettes, 'o-', color='steelblue')
ax.axvline(x=best_k, color='coral', linestyle='--', label=f'Best k={best_k}')
ax.set_xlabel('k'); ax.set_ylabel('Silhouette Score')
ax.set_title('K-Means — Optimal Cluster Count')
ax.legend()
plt.show()
print(f'Best k={best_k}')

In [ ]:
# Cluster map
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(coords)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
scatter = axes[0].scatter(df['lon'], df['lat'], c=df['cluster'], cmap='tab10', alpha=0.5, s=8)
axes[0].set_title(f'Business Clusters (k={best_k})')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

colors = df['is_expired'].map({0: 'steelblue', 1: 'coral'})
axes[1].scatter(df['lon'], df['lat'], c=colors, alpha=0.5, s=8)
axes[1].set_title('Active (Blue) vs Expired (Red)')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# Survival by cluster
cluster_stats = df.groupby('cluster').agg(
    count=('is_expired', 'size'),
    survival_rate=('is_expired', lambda x: (1 - x.mean()) * 100),
    avg_nearby=('nearby_businesses', 'mean'),
).round(1)
print(cluster_stats)

fig, ax = plt.subplots(figsize=(8, 4))
cluster_stats['survival_rate'].plot(kind='bar', color='steelblue', ax=ax)
ax.axhline(y=(1 - df['is_expired'].mean()) * 100, color='coral', linestyle='--',
           label=f'Overall: {(1 - df["is_expired"].mean()) * 100:.1f}%')
ax.set_ylabel('Survival Rate (%)')
ax.set_title('Survival Rate by Geographic Cluster')
ax.legend()
plt.show()

In [ ]:
# Density vs survival — tests the hypothesis
density_survival = df.groupby(
    pd.cut(df['nearby_businesses'], bins=8)
)['is_expired'].apply(lambda x: (1 - x.mean()) * 100)

fig, ax = plt.subplots(figsize=(10, 5))
density_survival.plot(kind='bar', color='steelblue', ax=ax)
ax.set_xlabel('Nearby Businesses (within ~200m)')
ax.set_ylabel('Survival Rate (%)')
ax.set_title('Business Density vs Survival Rate')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Hypothesis: businesses in denser clusters are more successful.')
print('Result: moderate support — denser areas show slightly higher survival,')
print('but the relationship is not strong or monotonic.')

## Part 5: Spatial Model

Adds GPS-derived features (**nearby business count**, **distance to MBTA**) to the entrepreneur model to see if spatial features improve prediction.

In [ ]:
from modeling import train_and_evaluate

le_n2, le_c2 = LabelEncoder(), LabelEncoder()
df_sp = df.copy()
df_sp['n_enc'] = le_n2.fit_transform(df_sp['neighborhood'])
df_sp['c_enc'] = le_c2.fit_transform(df_sp['business_category'])

feat_cols = ['n_enc', 'c_enc', 'nearby_businesses']
feat_names_sp = ['Neighborhood', 'Business Category', 'Nearby Businesses']
if 'dist_to_mbta' in df_sp.columns:
    feat_cols.append('dist_to_mbta')
    feat_names_sp.append('Dist to MBTA (km)')

X_sp = df_sp[feat_cols].values
y_sp = df_sp['is_expired'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_sp, y_sp, test_size=0.2, random_state=42, stratify=y_sp)
sp_results = train_and_evaluate(X_tr, X_te, y_tr, y_te, feat_names_sp)

In [ ]:
# Compare entrepreneur vs spatial model
ent_best_f1 = max(ent_results.values(), key=lambda x: x['f1'])['f1']
sp_best_f1 = max(sp_results.values(), key=lambda x: x['f1'])['f1']

print(f'Entrepreneur model (neigh + type only):     F1 = {ent_best_f1:.4f}')
print(f'Spatial model (neigh + type + GPS features): F1 = {sp_best_f1:.4f}')
print(f'Improvement: {(sp_best_f1 - ent_best_f1):.4f}')
print()
print('GPS features provide a modest improvement, but the fundamental')
print('challenge remains: business success depends heavily on factors')
print('not captured in public registration data.')

## Summary

| Model | Best Algorithm | Key Metric | Interpretation |
|-------|---------------|------------|----------------|
| Full Classification | Decision Tree | 94.9% accuracy, 0.89 CV F1 | Strong prediction, but uses features only known after opening |
| Entrepreneur Model | KNN | 66% accuracy, 0.26 F1 | Neighborhood + type alone are weak predictors |
| Longevity Regression | Random Forest | R²=0.03 | Location/type explain almost no variance in lifespan |
| K-Means Clustering | k=4 | Silhouette=0.60 | Clear geographic clusters; some variation in survival |
| Spatial Model | KNN | 66% accuracy, 0.30 F1 | GPS features improve slightly over entrepreneur model |

**Key finding**: Business success in Boston is *not* well predicted by neighborhood and business type alone. The survival heatmap reveals specific high/low-risk combinations, but the dominant factors in business longevity (financing, management quality, competition intensity) are not captured in public registration data.